# SI10-2026 | Ponderada | Análise de Sensibilidade em Métricas de Interface Digital

Nesta atividade, você vai analisar quais variáveis de uma interface digital têm maior impacto sobre a taxa de conversão.

A entrega deve ser feita neste notebook, com código, tabelas, gráficos e respostas curtas.

## Contexto

Uma equipe de produto quer decidir qual métrica de interface deve receber prioridade no próximo ciclo de melhoria.

Os dados representam observações diárias de um aplicativo de compras.

A métrica alvo é a taxa de conversão.

As variáveis de entrada são taxa de abandono do carrinho, profundidade média de scroll e tempo até o primeiro clique em produto.

## Preparação

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.precision", 3)

## Dados

Execute a célula abaixo para criar a base da atividade.

In [ ]:
rng = np.random.default_rng(42)
n_dias = 180

taxa_abandono = rng.normal(48, 8, n_dias).clip(25, 75)
profundidade_scroll = rng.normal(62, 12, n_dias).clip(25, 95)
tempo_primeiro_clique = rng.normal(7, 2.2, n_dias).clip(2, 15)

ruido = rng.normal(0, 0.35, n_dias)
taxa_conversao = (
    7.5
    - 0.055 * taxa_abandono
    + 0.026 * profundidade_scroll
    - 0.085 * tempo_primeiro_clique
    + ruido
).clip(0.5, 9.0)

df = pd.DataFrame({
    "data": pd.date_range("2026-01-01", periods=n_dias, freq="D"),
    "taxa_abandono_carrinho_pct": taxa_abandono,
    "profundidade_scroll_pct": profundidade_scroll,
    "tempo_primeiro_clique_s": tempo_primeiro_clique,
    "taxa_conversao_pct": taxa_conversao,
})

df.head()

,data,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
0,2026-01-01,50.438,77.672,6.664,5.589
1,2026-01-02,39.680,64.633,7.843,6.042
2,2026-01-03,54.004,57.069,9.200,5.318
3,2026-01-04,55.525,75.275,4.671,5.944
4,2026-01-05,32.392,67.145,6.725,6.804


In [ ]:
features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
]
target = "taxa_conversao_pct"

## Parte 1: Exploração

Crie ao menos um gráfico ou tabela para investigar a relação entre as variáveis de entrada e a taxa de conversão.

In [ ]:
# Use esta célula para criar sua análise exploratória.

colunas_numericas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
    "taxa_conversao_pct",
]

df[colunas_numericas].corr()

,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
taxa_abandono_carrinho_pct,1.000,-0.068,-0.116,-0.643
profundidade_scroll_pct,-0.068,1.000,0.050,0.485
tempo_primeiro_clique_s,-0.116,0.050,1.000,-0.229
taxa_conversao_pct,-0.643,0.485,-0.229,1.000


In [ ]:
# Preencha com uma variável de entrada para visualizar.
# Use exatamente um dos nomes que aparecem em features.

variavel_x = "taxa_abandono_carrinho_pct"

if variavel_x not in features:
    raise ValueError("Preencha variavel_x com uma variável da lista features.")

fig = px.scatter(
    df,
    x=variavel_x,
    y="taxa_conversao_pct",
    trendline="ols",
    title="Relação com a taxa de conversão",
)
fig.show()

Escreva quais duas variáveis você escolheu para a análise de sensibilidade e justifique com evidências da exploração.

**Resposta:**

Escolhi `taxa_abandono_carrinho_pct` e `profundidade_scroll_pct`.

A escolha vem da matriz de correlação com a `taxa_conversao_pct`. O abandono do carrinho tem r = -0,643, a correlação mais forte em módulo (negativa); a profundidade de scroll vem em seguida, com r = +0,485 (positiva, moderada); e o tempo até o primeiro clique fica bem atrás, com r = -0,229.

As duas que escolhi são as que mais se movem junto com a conversão, então tendem a render o maior ganho por esforço de melhoria. O tempo até o primeiro clique fica de fora por ter a associação mais fraca. O sinal de cada correlação já antecipa a direção do efeito: menos abandono leva a mais conversão, e mais scroll também.

In [ ]:
# Suporte à escolha: força da correlação (em módulo) com a taxa de conversão
corr_alvo = (
    df[features + [target]]
    .corr()[target]
    .drop(target)
    .rename("correlacao")
    .to_frame()
)
corr_alvo["abs_correlacao"] = corr_alvo["correlacao"].abs()
corr_alvo = corr_alvo.sort_values("abs_correlacao", ascending=False)
print(corr_alvo.round(3))

fig = px.bar(
    corr_alvo.reset_index(),
    x="abs_correlacao",
    y="index",
    orientation="h",
    title="Força da correlação (|r|) de cada variável com a taxa de conversão",
    labels={"abs_correlacao": "|correlação|", "index": ""},
)
fig.update_yaxes(autorange="reversed")
fig.show()

                            correlacao  abs_correlacao
taxa_abandono_carrinho_pct      -0.643           0.643
profundidade_scroll_pct          0.485           0.485
tempo_primeiro_clique_s         -0.229           0.229


## Parte 2: Modelo

Ajuste o modelo abaixo para estimar a taxa de conversão a partir das variáveis de entrada.

In [ ]:
X = df[features].to_numpy()
y = df[target].to_numpy()

X_design = np.column_stack([np.ones(len(X)), X])

coeficientes, *_ = np.linalg.lstsq(X_design, y, rcond=None)

pred = X_design @ coeficientes
erro = y - pred

mae = np.mean(np.abs(erro))
rmse = np.sqrt(np.mean(erro ** 2))

pd.DataFrame({
    "métrica": ["MAE", "RMSE"],
    "valor": [mae, rmse],
})

,métrica,valor
0,MAE,0.276
1,RMSE,0.344


Interprete o erro do modelo em relação à taxa de conversão.

**Resposta:**

O modelo linear erra, em média, 0,276 p.p. (MAE) e 0,344 p.p. (RMSE). Como a conversão média é de 5,87% (com desvio de 0,64 p.p.), esse erro médio equivale a cerca de 4,7% da média, o que é baixo para a escala da métrica.

O RMSE ser maior que o MAE indica que existem alguns dias com erro acima do típico, já que o RMSE penaliza mais os erros grandes. O R² de aproximadamente 0,71 mostra que as três variáveis explicam cerca de 71% da variação da conversão, e os 29% restantes ficam por conta do ruído do dia a dia.

No fim, o modelo é confiável o suficiente para orientar a análise de sensibilidade, ou seja, entender quem mais move a conversão, mas não para prever com precisão o valor exato de cada dia.

In [ ]:
# Erro em escala relativa + R²
ss_res = np.sum(erro ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

pd.DataFrame({
    "métrica": ["MAE (p.p.)", "RMSE (p.p.)", "MAE / média (%)", "RMSE / desvio", "R²"],
    "valor": [mae, rmse, mae / y.mean() * 100, rmse / y.std(), r2],
})

,métrica,valor
0,MAE (p.p.),0.276
1,RMSE (p.p.),0.344
2,MAE / média (%),4.699
3,RMSE / desvio,0.534
4,R²,0.714


## Parte 3: Análise de Sensibilidade

Calcule a sensibilidade para duas variáveis de entrada usando uma variação de 10%.

Use a fórmula: sensibilidade igual à variação percentual da saída dividida pela variação percentual da entrada.

In [ ]:
def prever_linha(linha):
    entrada = np.array([1] + [linha[feature] for feature in features])
    return float(entrada @ coeficientes)


linha_base = df[features].mean().to_dict()
saida_base = prever_linha(linha_base)

linha_base, saida_base

({'taxa_abandono_carrinho_pct': 47.54587889307049,
  'profundidade_scroll_pct': 62.44918010647032,
  'tempo_primeiro_clique_s': 6.9708957453209095},
 5.868747841831934)

In [ ]:
# Preencha com duas variáveis escolhidas na Parte 1.
# Use exatamente os nomes que aparecem em features.

variaveis_escolhidas = ["taxa_abandono_carrinho_pct", "profundidade_scroll_pct"]

if len(variaveis_escolhidas) != 2:
    raise ValueError("Preencha variaveis_escolhidas com duas variáveis da lista features.")

variaveis_invalidas = [v for v in variaveis_escolhidas if v not in features]

if variaveis_invalidas:
    raise ValueError(f"Variáveis fora de features: {variaveis_invalidas}")

variacao_entrada = 0.10

resultados = []

for variavel in variaveis_escolhidas:
    linha_cenario = linha_base.copy()
    valor_original = linha_base[variavel]
    valor_alterado = valor_original * (1 + variacao_entrada)
    linha_cenario[variavel] = valor_alterado

    saida_nova = prever_linha(linha_cenario)
    variacao_saida = (saida_nova - saida_base) / saida_base
    indice_sensibilidade = variacao_saida / variacao_entrada

    resultados.append({
        "variável": variavel,
        "valor_original": valor_original,
        "valor_alterado": valor_alterado,
        "saída_original": saida_base,
        "saída_nova": saida_nova,
        "variação_saida_pct": variacao_saida * 100,
        "índice_sensibilidade": indice_sensibilidade,
    })

tabela_sensibilidade = pd.DataFrame(resultados)
tabela_sensibilidade

,variável,valor_original,valor_alterado,saída_original,saída_nova,variação_saida_pct,índice_sensibilidade
0,taxa_abandono_carrinho_pct,47.546,52.300,5.869,5.582,-4.891,-0.489
1,profundidade_scroll_pct,62.449,68.694,5.869,6.020,2.578,0.258


Compare os índices de sensibilidade e indique qual variável tem maior impacto sobre a taxa de conversão.

Mostre o raciocínio: cite os valores da tabela e explique o que eles significam para a decisão.

**Resposta:**

Aplicando um aumento de 10% em cada entrada, a partir do ponto médio, o abandono do carrinho deu variação de saída de -4,89% (índice -0,489) e a profundidade de scroll deu +2,58% (índice +0,258).

O maior impacto é do abandono do carrinho. Em módulo, 0,489 é maior que 0,258, ou seja, ele pesa cerca de 1,9 vez mais sobre a conversão do que o scroll.

Partindo do ponto médio, o modelo prevê uma conversão de 5,87%. Quando aumento o abandono em 10% (de 47,5% para 52,3%), a conversão cai para 5,58%, uma queda de 4,89%. Já o mesmo aumento de 10% no scroll (de 62,4% para 68,7%) faz a conversão subir para 6,02%, um ganho de 2,58%. Os sinais batem com o que eu imaginava: o índice negativo do abandono mostra que mexer nessa variável para cima piora a conversão, ou seja, é reduzindo o abandono que se ganha conversão. E como esse é o índice de maior módulo, é onde a interface responde mais forte.

In [ ]:
# Comparação visual dos índices de sensibilidade
fig = px.bar(
    tabela_sensibilidade,
    x="variável",
    y="índice_sensibilidade",
    title="Índice de sensibilidade por variável (choque de +10%)",
    text="índice_sensibilidade",
    color="variável",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.add_hline(y=0, line_dash="dot")
fig.show()

## Parte 4: Decisão

Recomende uma ação de produto ou interface com base na análise.

Sua recomendação deve citar os números da tabela de sensibilidade.

**Resposta:**

Priorizar a redução da `taxa_abandono_carrinho_pct` no próximo ciclo de melhoria.

Vejo que essa ação é a mais relevante, porque é a alavanca de maior sensibilidade: índice -0,489, contra +0,258 do scroll, quase 1,9 vez mais impacto. Como o efeito é negativo, reduzir o abandono aumenta a conversão. Pelo cenário simulado, reduzir o abandono em 10% (de ~47,5% para ~42,8%) eleva a conversão de 5,87% para ~6,16%, um ganho de 0,287 p.p. (+4,89%), o maior retorno por esforço entre as variáveis analisadas.

Na prática, isso aponta para ações como encurtar o checkout, antecipar frete e impostos, oferecer mais meios de pagamento e disparar recuperação de carrinho. A profundidade de scroll (índice +0,258) fica como segunda prioridade, com melhorias de layout e conteúdo que incentivem a rolagem. O tempo até o primeiro clique (índice -0,112) considero o menos prioritário.

Aponte uma limitação, risco ou hipótese da sua análise.

**Resposta:**

A maior fragilidade desse notebook é ter assumido que a relação é linear, e isso provavelmente não vale na realidade. Acredito que cortar o abandono de 50% para 45% não deve dar o mesmo resultado que cortar de 30% para 25%, porque uma hora o efeito satura.

Outra limitação é que calculei mexendo numa variável de cada vez e deixando as outras quietas. Só que abandono, scroll e tempo de clique não vivem isolados, então o impacto real do abandono pode mudar quando ele anda junto com as outras variáveis.

In [ ]:
# Cenário de melhoria: reduzir o abandono do carrinho em 10%
linha_melhoria = linha_base.copy()
linha_melhoria["taxa_abandono_carrinho_pct"] *= 0.90
saida_melhoria = prever_linha(linha_melhoria)

print(f"Conversão base:           {saida_base:.3f}%")
print(f"Abandono -10% ({linha_base['taxa_abandono_carrinho_pct']:.1f}% -> {linha_melhoria['taxa_abandono_carrinho_pct']:.1f}%): {saida_melhoria:.3f}%")
print(f"Ganho absoluto:           {saida_melhoria - saida_base:+.3f} p.p.")
print(f"Ganho relativo:           {(saida_melhoria - saida_base) / saida_base * 100:+.2f}%")

Conversão base:           5.869%
Abandono -10% (47.5% -> 42.8%): 6.156%
Ganho absoluto:           +0.287 p.p.
Ganho relativo:           +4.89%


## Ao Além dos Aléns

Faça uma simulação de Monte Carlo para estimar como a taxa de conversão pode variar sob incerteza nas variáveis de entrada.

In [ ]:
# Use esta célula para sua simulação.

n_simulacoes = 1000

amostras = pd.DataFrame({
    "taxa_abandono_carrinho_pct": rng.normal(
        linha_base["taxa_abandono_carrinho_pct"], 5, n_simulacoes
    ).clip(25, 75),
    "profundidade_scroll_pct": rng.normal(
        linha_base["profundidade_scroll_pct"], 8, n_simulacoes
    ).clip(25, 95),
    "tempo_primeiro_clique_s": rng.normal(
        linha_base["tempo_primeiro_clique_s"], 1.5, n_simulacoes
    ).clip(2, 15),
})

amostras_design = np.column_stack([
    np.ones(len(amostras)),
    amostras[features].to_numpy(),
])
previsoes = amostras_design @ coeficientes

pd.Series(previsoes).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])

count    1000.000
mean        5.869
std         0.393
min         4.654
10%         5.369
25%         5.617
50%         5.854
75%         6.131
90%         6.379
max         7.193
dtype: float64

In [ ]:
fig = px.histogram(
    pd.DataFrame({"taxa_conversao_pct_prevista": previsoes}),
    x="taxa_conversao_pct_prevista",
    nbins=30,
    title="Distribuição simulada da taxa de conversão",
)
fig.show()

Interprete o que a distribuição simulada indica sobre o risco da sua recomendação.

**Resposta:**

A simulação de Monte Carlo, com 1.000 cenários variando as três entradas dentro de faixas plausíveis, gera uma distribuição da conversão com média em torno de 5,87% e desvio de 0,39 p.p. O P10 fica em 5,37% e o P90 em 6,38%, ou seja, em 80% dos cenários a conversão fica entre cerca de 5,4% e 6,4%. A chance de cair abaixo de 5% é de apenas ~1%, e abaixo de 5,5% é de ~17,5%.

A chance de uma queda forte, abaixo de 5%, é pequena, e a distribuição fica razoavelmente simétrica em torno da média. Isso me deixa mais confiante na ideia de atacar o abandono: mesmo nos piores cenários da simulação, a conversão quase nunca despenca. Por outro lado, por mais estreita que a distribuição pareça, a simulação só espalha a incerteza das entradas e roda tudo pela mesma equação ajustada, então ela não consegue me dizer se a relação é mesmo de causa e efeito.

In [ ]:
# Probabilidades de risco a partir da simulação
serie_prev = pd.Series(previsoes)

print(serie_prev.describe(percentiles=[0.05, 0.1, 0.5, 0.9, 0.95]).round(3))
print()
for limite in [5.0, 5.5, 6.0]:
    p = (serie_prev < limite).mean() * 100
    print(f"P(conversão < {limite:.1f}%) = {p:.1f}%")
print(f"\nIntervalo central de 80% (P10–P90): "
      f"{serie_prev.quantile(0.1):.2f}% a {serie_prev.quantile(0.9):.2f}%")

count    1000.000
mean        5.869
std         0.393
min         4.654
5%          5.221
10%         5.369
50%         5.854
90%         6.379
95%         6.539
max         7.193
dtype: float64

P(conversão < 5.0%) = 1.0%
P(conversão < 5.5%) = 17.5%
P(conversão < 6.0%) = 64.2%

Intervalo central de 80% (P10–P90): 5.37% a 6.38%


## Política de Uso de IA

O uso de IA é permitido para apoio técnico, revisão de texto e estudo dos conceitos.

As escolhas de variáveis, os cálculos, a comparação dos índices e a recomendação devem refletir sua análise dos resultados deste notebook.

Você deve ser capaz de explicar qualquer resposta entregue.

Respostas sem relação com os números gerados, com indícios de cópia ou que não possam ser justificadas poderão ser tratadas como fora da proposta.

## Instruções de entrega

A entrega deverá ser feita no GitHub ou no próprio Google Colab.

Links **sem permissão** de acesso terão um desconto de 20% na nota.